# SmartBay Dataset - FiftyOne Import and Analysis

This notebook imports the SmartBay YOLO dataset from `C:\\SmartBay\\DataSets\\Original`, preserves the original annotations, creates a separate fish-only annotation field, audits image quality and video-sequence leakage, writes CSV reports, and opens the FiftyOne App.

Run the cells in order. The notebook assumes that you are using the same Python environment in which FiftyOne was installed for the Fish4Knowledge project.

## 1. Configuration

Original images and YOLO labels belong under `C:\SmartBay\DataSets\Original`. Reports are saved under `C:\SmartBay\Analysis\Original`. Move the complete Original folder before running this notebook. Existing FiftyOne image paths are updated from the previous E: or C: Original locations, preserving annotations and tags.

Keep `REBUILD_DATASET = False` to preserve your reviews. This notebook analyses Original; the multi-dataset notebook remains a separate file.


In [ ]:
from pathlib import Path

SMARTBAY_ROOT = Path(r"C:\SmartBay")
DATASET_FOLDER_NAME = "Original"
DATASETS_ROOT = SMARTBAY_ROOT / "DataSets"
ORIGINAL_DATA_ROOT = DATASETS_ROOT / DATASET_FOLDER_NAME
ANALYSIS_ROOT = SMARTBAY_ROOT / "Analysis"
DATASET_NAME = "smartbay_capstone"
REBUILD_DATASET = False
APP_PORT = 5151

ORIGINAL_LABEL_FIELD = "ground_truth"
FISH_LABEL_FIELD = "fish_detections"

NON_FISH_CLASSES = {
    "Aurelia_aurita",       # jellyfish
    "Brachyura",            # crabs
    "Maja_brachydactyla",  # spider crab
    "Phoca_vitulina",       # harbour seal
    "Starfish",
}

print(f"SmartBay root: {SMARTBAY_ROOT}")
print(f"FiftyOne dataset: {DATASET_NAME}")

## 2. Imports and helper functions

The original `ground_truth` field is never altered. A new `fish_detections` field is created without the known non-fish classes. Individual fish classes remain unchanged.

In [ ]:
import csv
import re
from collections import Counter, defaultdict

import cv2
import fiftyone as fo
from fiftyone import ViewField as F
import numpy as np
import yaml


def find_data_yaml(root):
    direct = root / "data.yaml"
    if direct.exists():
        return direct
    candidates = sorted(root.rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError(
            f"No data.yaml found under {root}. Extract the SmartBay ZIP first."
        )
    if len(candidates) > 1:
        print(f"Multiple data.yaml files found; using {candidates[0]}")
    return candidates[0]


def read_yaml(path):
    with path.open("r", encoding="utf-8") as stream:
        config = yaml.safe_load(stream)
    if not isinstance(config, dict):
        raise ValueError(f"Invalid YOLO YAML: {path}")
    return config


def find_splits(dataset_dir, config):
    found = []
    for split, yaml_key in (("train", "train"), ("valid", "val"), ("test", "test")):
        if config.get(yaml_key) or (dataset_dir / split / "images").exists():
            found.append(split)
    return found


def split_from_tags(tags):
    for split in ("train", "valid", "test"):
        if split in tags:
            return split
    return "unknown"


def source_sequence_id(filepath):
    name = Path(filepath).name
    name = re.sub(r"\.rf\.[0-9a-f]+(?=\.[^.]+$)", "", name, flags=re.I)
    stem = Path(name).stem
    match = re.match(r"(.+?)_frame_\d+", stem, flags=re.I)
    if match:
        return match.group(1)
    return f"still::{stem}"


def filtered_fish_detections(detections):
    if detections is None:
        return fo.Detections(detections=[])
    retained = []
    for detection in detections.detections:
        if detection.label in NON_FISH_CLASSES:
            continue
        retained.append(
            fo.Detection(
                label=detection.label,
                bounding_box=list(detection.bounding_box),
                confidence=detection.confidence,
            )
        )
    return fo.Detections(detections=retained)


def image_quality(filepath):
    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if image is None:
        return float("nan"), float("nan"), float("nan")
    brightness = float(np.mean(image))
    contrast = float(np.std(image))
    blur_score = float(cv2.Laplacian(image, cv2.CV_64F).var())
    return brightness, contrast, blur_score


print(f"FiftyOne version: {fo.__version__}")

## 3. Locate and import the YOLO dataset

The notebook searches only through `C:\\SmartBay\\DataSets\\Original`, so it works whether `data.yaml` is directly in that folder or inside the extracted `smartbay.v115i.yolov8` folder.

In [ ]:
if not SMARTBAY_ROOT.exists():
    raise FileNotFoundError(f"Folder does not exist: {SMARTBAY_ROOT}")

if not ORIGINAL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Original data folder does not exist: {ORIGINAL_DATA_ROOT}")

DATA_YAML = find_data_yaml(ORIGINAL_DATA_ROOT)
DATASET_DIR = DATA_YAML.parent
YOLO_CONFIG = read_yaml(DATA_YAML)
SPLITS = find_splits(DATASET_DIR, YOLO_CONFIG)

print(f"data.yaml: {DATA_YAML}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Splits found: {SPLITS}")
print(f"Classes declared: {YOLO_CONFIG.get('nc', len(YOLO_CONFIG.get('names', [])))}")

if not SPLITS:
    raise RuntimeError("No train, valid or test images were found")

# Remove an empty dataset that may have been left by an interrupted import.
if fo.dataset_exists(DATASET_NAME) and len(fo.load_dataset(DATASET_NAME)) == 0:
    print(f"Removing empty dataset left by an earlier failed import: {DATASET_NAME}")
    fo.delete_dataset(DATASET_NAME)

if fo.dataset_exists(DATASET_NAME) and REBUILD_DATASET:
    print(f"Deleting existing dataset: {DATASET_NAME}")
    fo.delete_dataset(DATASET_NAME)

if fo.dataset_exists(DATASET_NAME):
    dataset = fo.load_dataset(DATASET_NAME)
    # Preserve annotations, tags and audit fields when relocating existing samples.
    old_roots = ("E:/SmartBay/Original", "C:/SmartBay/Original")
    filepaths = dataset.values("filepath")
    relocated = []
    changed = 0
    for filepath in filepaths:
        normalized = filepath.replace("\\", "/")
        new_path = Path(filepath)
        for old_root in old_roots:
            if normalized.lower().startswith(old_root.lower() + "/"):
                new_path = ORIGINAL_DATA_ROOT / normalized[len(old_root) + 1:]
                changed += 1
                break
        relocated.append(str(new_path))
    missing = [p for p in relocated if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} dataset images are missing after relocation. "
            f"First missing image: {missing[0]}. Check that the full folder was copied."
        )
    if changed:
        dataset.set_values("filepath", relocated)
        print(f"Updated {changed:,} image paths to {ORIGINAL_DATA_ROOT}")
    print(f"Opened existing dataset with {len(dataset):,} images")
else:
    dataset = fo.Dataset(DATASET_NAME, persistent=True)
    for split in SPLITS:
        importer_split = "val" if split == "valid" else split
        print(f"Importing {split} (YOLO key: {importer_split})...")
        dataset.add_dir(
            dataset_dir=str(DATASET_DIR),
            yaml_path=str(DATA_YAML),
            dataset_type=fo.types.YOLOv5Dataset,
            split=importer_split,
            tags=[split],
            label_field=ORIGINAL_LABEL_FIELD,
        )
    print(f"Imported {len(dataset):,} images")

dataset

## 4. Calculate image, annotation and video-sequence fields

This is the longest cell. For all 7,131 images it may take several minutes on the first run. The values are stored in the persistent FiftyOne dataset.

In [ ]:
dataset.compute_metadata(overwrite=False)
sequence_splits = defaultdict(set)
total = len(dataset)

for index, sample in enumerate(dataset.iter_samples(autosave=True), start=1):
    split = split_from_tags(sample.tags)
    sequence = source_sequence_id(sample.filepath)
    original = sample[ORIGINAL_LABEL_FIELD]
    fish = filtered_fish_detections(original)
    original_labels = [d.label for d in original.detections] if original else []
    brightness, contrast, blur_score = image_quality(sample.filepath)

    sequence_splits[sequence].add(split)
    sample["split"] = split
    sample["source_sequence"] = sequence
    sample["brightness"] = brightness
    sample["contrast"] = contrast
    sample["blur_laplacian"] = blur_score
    sample["original_box_count"] = len(original_labels)
    sample["fish_box_count"] = len(fish.detections)
    sample["contains_non_fish"] = any(
        label in NON_FISH_CLASSES for label in original_labels
    )
    sample[FISH_LABEL_FIELD] = fish

    if index % 250 == 0 or index == total:
        print(f"Processed {index:,}/{total:,}")

crossing_sequences = {
    sequence for sequence, splits in sequence_splits.items() if len(splits) > 1
}
dataset.set_values(
    "sequence_crosses_splits",
    [s in crossing_sequences for s in dataset.values("source_sequence")],
)

print(f"Source groups: {len(sequence_splits):,}")
print(f"Groups crossing splits: {len(crossing_sequences):,}")

## 5. Create saved FiftyOne views

These views will appear in FiftyOne and can be opened again in later sessions.

In [ ]:
review_count = max(1, min(250, round(len(dataset) * 0.05)))
views = {
    "01_train": dataset.match_tags("train"),
    "02_validation": dataset.match_tags("valid"),
    "03_test": dataset.match_tags("test"),
    "04_contains_non_fish": dataset.match(F("contains_non_fish") == True),
    "05_no_fish_boxes": dataset.match(F("fish_box_count") == 0),
    "06_sequence_split_leakage": dataset.match(F("sequence_crosses_splits") == True),
    "07_darkest_5_percent": dataset.sort_by("brightness").limit(review_count),
    "08_lowest_contrast_5_percent": dataset.sort_by("contrast").limit(review_count),
    "09_blurriest_5_percent": dataset.sort_by("blur_laplacian").limit(review_count),
}

existing_views = set(dataset.list_saved_views())
for name, view in views.items():
    if name in existing_views:
        dataset.delete_saved_view(name)
    dataset.save_view(name, view)

print("Saved views:")
for name in dataset.list_saved_views():
    print(f"  {name}")

## 6. Write class and sequence-audit reports

In [ ]:
REPORT_DIR = ANALYSIS_ROOT / DATASET_FOLDER_NAME
REPORT_DIR.mkdir(parents=True, exist_ok=True)

sequence_counts = defaultdict(Counter)
for sequence, split in zip(
    dataset.values("source_sequence"), dataset.values("split")
):
    sequence_counts[sequence][split] += 1

sequence_report = REPORT_DIR / "smartbay_sequence_split_audit.csv"
with sequence_report.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["source_sequence", "train", "valid", "test", "crosses_splits"])
    for sequence in sorted(sequence_counts):
        counts = sequence_counts[sequence]
        writer.writerow([
            sequence, counts["train"], counts["valid"], counts["test"],
            len(sequence_splits[sequence]) > 1,
        ])

class_counts = Counter(
    dataset.values(f"{ORIGINAL_LABEL_FIELD}.detections.label", unwind=True)
)
class_report = REPORT_DIR / "smartbay_class_counts.csv"
with class_report.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["class", "bounding_boxes", "retained_as_fish"])
    for label, count in class_counts.most_common():
        writer.writerow([label, count, label not in NON_FISH_CLASSES])

print(f"Reports saved in: {REPORT_DIR}")
print(f"  {sequence_report.name}")
print(f"  {class_report.name}")

## 7. Display the audit summary

In [ ]:
print("SMARTBAY AUDIT SUMMARY")
print("=" * 60)
print(f"Images:                 {len(dataset):,}")
print(f"Original boxes:         {sum(class_counts.values()):,}")
print(f"Fish boxes retained:    {sum(dataset.values('fish_box_count')):,}")
print(f"Original classes:       {len(class_counts):,}")
print(f"Fish classes retained:  {sum(1 for c in class_counts if c not in NON_FISH_CLASSES):,}")
print(f"Source groups:          {len(sequence_splits):,}")
print(f"Groups crossing splits: {len(crossing_sequences):,}")

print("\nImages by split:")
for split in ("train", "valid", "test", "unknown"):
    count = len(dataset.match(F("split") == split))
    if count:
        print(f"  {split:8s} {count:,}")

print("\nNon-fish boxes excluded from fish_detections:")
for label in sorted(NON_FISH_CLASSES):
    print(f"  {label:24s} {class_counts.get(label, 0):,}")

print("\nMost frequent retained fish classes:")
fish_class_counts = Counter({
    label: count for label, count in class_counts.items()
    if label not in NON_FISH_CLASSES
})
for label, count in fish_class_counts.most_common(15):
    print(f"  {label:24s} {count:,}")

In [ ]:

session = fo.launch_app(dataset, port=APP_PORT, auto=False)
session.open_tab()
print(f"FiftyOne is available at http://localhost:{APP_PORT}")

ValueError: `dataset` must be a <class 'fiftyone.core.dataset.Dataset'> or None; found <class 'str'>

## 8. Open FiftyOne

The app should open in your browser. Select either `ground_truth` or `fish_detections` in the FiftyOne sidebar to compare the original and filtered annotations.